In [4]:
from google.colab import files
uploaded = files.upload()

Saving dogs.zip to dogs.zip


In [6]:
import zipfile
with zipfile.ZipFile("dogs.zip", "r") as zip_ref:
    zip_ref.extractall(".")
print("Done!")

Done!


In [7]:
import os
for root, dirs, files in os.walk("dogs"):
    print(root, "→", len(files), "files")

dogs → 0 files
dogs/val → 0 files
dogs/val/lily → 7 files
dogs/val/henry → 15 files
dogs/val/charlotte → 11 files
dogs/val/ivy → 9 files
dogs/val/wilbur → 7 files
dogs/train → 0 files
dogs/train/lily → 91 files
dogs/train/henry → 83 files
dogs/train/charlotte → 79 files
dogs/train/ivy → 87 files
dogs/train/wilbur → 63 files


In [10]:
!pip install pillow-heif

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 82.2 MB/s eta 0:00:00


In [11]:
from pillow_heif import register_heif_opener
from PIL import Image
import os

register_heif_opener()

converted = 0
for root, dirs, files in os.walk("dogs"):
    for file in files:
        if file.lower().endswith(".heic"):
            heic_path = os.path.join(root, file)
            jpg_path = os.path.splitext(heic_path)[0] + ".jpg"
            img = Image.open(heic_path)
            img.save(jpg_path, "JPEG")
            os.remove(heic_path)
            converted += 1

print(f"Converted {converted} files!")

Converted 442 files!


In [12]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np

print("TensorFlow version:", tf.__version__)
print("Ready to go!")

TensorFlow version: 2.19.0
Ready to go!


In [13]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2]
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    "dogs/train",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical"
)

val_data = val_datagen.flow_from_directory(
    "dogs/val",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical"
)

print("\nDog name → number mapping:")
print(train_data.class_indices)

Found 403 images belonging to 5 classes.
Found 49 images belonging to 5 classes.

Dog name → number mapping:
{'charlotte': 0, 'henry': 1, 'ivy': 2, 'lily': 3, 'wilbur': 4}


In [14]:
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(5, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    train_data,
    epochs=15,
    validation_data=val_data
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 63s 4s/step - accuracy: 0.2382 - loss: 1.8660 - val_accuracy: 0.2653 - val_loss: 1.5472
Epoch 2/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 54s 4s/step - accuracy: 0.4864 - loss: 1.2675 - val_accuracy: 0.3265 - val_loss: 1.4779
Epoch 3/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 50s 4s/step - accuracy: 0.6526 - loss: 1.0001 - val_accuracy: 0.3265 - val_loss: 1.5337
Epoch 4/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 50s 4s/step - accuracy: 0.7146 - loss: 0.8213 - val_accuracy: 0.4286 - val_loss: 1.3910
Epoch 5/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 52s 4s/step - accuracy: 0.7593 - loss: 0.6602 - val_accuracy: 0.4082 - val_loss: 1.4046
Epoch 6/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 49s 4s/step - accuracy: 0.8337 - loss: 0.5282 - val_accuracy: 0.3878 - val_loss: 1.5521
Epoch 7/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 49s 4s/step - accuracy: 0.8536 - loss: 0.4943 - val_accuracy: 0.4286 - val_loss: 1.4989
Epoch 8/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 52s 4s/step - accuracy: 0.8660 

In [15]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history2 = model.fit(
    train_data,
    epochs=15,
    validation_data=val_data
)

Epoch 1/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 72s 5s/step - accuracy: 0.7072 - loss: 0.8219 - val_accuracy: 0.4490 - val_loss: 1.5592
Epoch 2/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 53s 4s/step - accuracy: 0.7667 - loss: 0.6325 - val_accuracy: 0.4694 - val_loss: 1.5821
Epoch 3/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 54s 4s/step - accuracy: 0.8437 - loss: 0.4914 - val_accuracy: 0.4490 - val_loss: 1.6194
Epoch 4/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 53s 4s/step - accuracy: 0.8561 - loss: 0.4301 - val_accuracy: 0.4694 - val_loss: 1.6595
Epoch 5/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 52s 4s/step - accuracy: 0.8561 - loss: 0.4433 - val_accuracy: 0.4490 - val_loss: 1.7070
Epoch 6/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 54s 4s/step - accuracy: 0.8908 - loss: 0.3629 - val_accuracy: 0.4286 - val_loss: 1.7700
Epoch 7/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 52s 4s/step - accuracy: 0.8834 - loss: 0.3322 - val_accuracy: 0.4082 - val_loss: 1.8402
Epoch 8/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 53s 4s/step - accuracy: 0.8908 - loss: 0.3156 - val_accuracy: 0.3878 - val_loss:

In [18]:
import gradio as gr
import numpy as np
from tensorflow.keras.preprocessing import image

dog_names = {0: "Charlotte", 1: "Henry", 2: "Ivy", 3: "Lily", 4: "Wilbur"}

def identify_dog(img):
    img_resized = img.resize((224, 224))
    img_array = image.img_to_array(img_resized)
    img_array = np.expand_dims(img_array, axis=0) / 255.0
    predictions = model.predict(img_array)[0]
    return {dog_names[i]: float(predictions[i]) for i in range(5)}

app = gr.Interface(
    fn=identify_dog,
    inputs=gr.Image(type="pil", label="Upload a photo"),
    outputs=gr.Label(num_top_classes=5, label="Who is this?"),
    title="Dog Identifier",
    description="Upload a photo of Lily, Henry, Wilbur, Charlotte, or Ivy!"
)

app.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7dbb245d16d8dc8baa.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
